In [1]:
# =====================================================
# 0. 라이브러리 세팅
# =====================================================
def load_libraries():
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import scipy.stats as stats
    import math
    import platform
    import ast

    pd.set_option('display.float_format', "{:.2f}".format)

    if platform.system() == 'Darwin':
        plt.rcParams['font.family'] = 'AppleGothic'
    elif platform.system() == 'Windows':
        plt.rcParams['font.family'] = 'Malgun Gothic'
    else:
        plt.rcParams['font.family'] = 'NanumGothic'

    return pd, np, plt, ast


# =====================================================
# 1. 데이터 로드
# =====================================================
def load_data(pd):
    df_port = pd.read_csv("../dataset/portfolio.csv", index_col=0)
    df_prof = pd.read_csv("../dataset/profile.csv", index_col=0)
    df_tran = pd.read_csv("../dataset/transcript.csv", index_col=0)
    df_menu = pd.read_csv("../dataset/starbucks_menu_260112.csv", index_col=0)

    print("프로모션 제공 데이터 크기:", df_port.shape)
    print("고객정보 데이터 크기:", df_prof.shape)
    print("제공 프로모션 데이터 크기:", df_tran.shape)
    print("메뉴 정보 데이터 크기:", df_menu.shape)

    return df_port, df_prof, df_tran, df_menu


# =====================================================
# 2. Portfolio 전처리
# =====================================================
def preprocess_portfolio(df_port, np):
    df_port = df_port.copy()

    # One-hot encoding
    df_port['ch_web'] = df_port['channels'].apply(lambda x: 1 if 'web' in x else 0)
    df_port['ch_email'] = df_port['channels'].apply(lambda x: 1 if 'email' in x else 0)
    df_port['ch_mobile'] = df_port['channels'].apply(lambda x: 1 if 'mobile' in x else 0)
    df_port['ch_social'] = df_port['channels'].apply(lambda x: 1 if 'social' in x else 0)
    df_port['channel_count'] = df_port[['ch_web','ch_email','ch_mobile','ch_social']].sum(axis=1)

    # 필요없는 컬럼 제거
    drop_cols = ['channels', 'Unnamed: 0']
    df_port = df_port.drop(columns=[c for c in drop_cols if c in df_port.columns])

    # 파생 변수
    df_port['reward_ratio'] = np.where(df_port['difficulty']==0, df_port['reward'], df_port['reward']/df_port['difficulty'])
    df_port['offer_strength'] = df_port['reward'] - df_port['difficulty']

    # offer_id로 컬럼명 변경
    df_port = df_port.rename(columns={'id':'offer_id'})
    df_port['offer_id'] = df_port['offer_id'].astype(str)

    return df_port


# =====================================================
# 3. Transcript 전처리 + viewed_before_complete 플래그
# =====================================================
def preprocess_transcript(df_tran, ast):
    df_tran = df_tran.copy()

    # value 컬럼 파싱
    df_tran['value'] = df_tran['value'].apply(ast.literal_eval)
    df_tran['amount'] = df_tran['value'].str.get('amount')
    df_tran['reward'] = df_tran['value'].str.get('reward')

    temp_id1 = df_tran['value'].str.get('offer id')
    temp_id2 = df_tran['value'].str.get('offer_id')
    df_tran['offer_id'] = temp_id1.fillna(temp_id2)

    # 컬럼 제거, day 파생
    df_tran.drop(columns=['value','reward'], inplace=True)
    df_tran['day'] = df_tran['time'] // 24

    # ID 타입 문자열로
    df_tran['offer_id'] = df_tran['offer_id'].astype(str)
    df_tran['customer_id'] = df_tran['person'].astype(str)

    # ==========================
    # viewed 없이 completed 이벤트 플래그
    # ==========================
    viewed_set = set(df_tran[df_tran['event']=='offer viewed'][['person','offer_id']].apply(tuple, axis=1))
    def _viewed_flag(log):
        if log['event'] != 'offer completed':
            return None
        return 1 if (log['person'], log['offer_id']) in viewed_set else 0

    df_tran['viewed_before_complete'] = df_tran.apply(_viewed_flag, axis=1)

    unaware = df_tran[df_tran['viewed_before_complete']==0].shape[0]
    total_completed = df_tran[df_tran['event']=='offer completed'].shape[0]
    print(f"이벤트 미인지 완료: {unaware}건 / 전체 완료: {total_completed}건 ({unaware/total_completed*100:.1f}%)")

    print("✅ df_tran 전처리 완료")
    return df_tran


# =====================================================
# 4. Profile 전처리
# =====================================================
def preprocess_profile(df_prof, pd, np):
    df_prof = df_prof.copy()

    # 이상치 처리
    df_prof['age'] = df_prof['age'].replace(118, np.nan)
    df_prof['gender'] = df_prof['gender'].fillna('Unknown')
    df_prof['income'] = df_prof['income'].fillna(0)

    # 결측 그룹 flag
    df_prof['is_profile_missing'] = np.where(
        (df_prof['gender']=='Unknown') & (df_prof['income']==0) & (df_prof['age'].isna()), 1, 0
    )

    # 날짜 변환
    df_prof['became_member_on'] = pd.to_datetime(df_prof['became_member_on'], format='%Y%m%d')

    # id → customer_id
    df_prof = df_prof.rename(columns={'id':'customer_id'})
    df_prof['customer_id'] = df_prof['customer_id'].astype(str)

    return df_prof


# =====================================================
# 5. 병합 (tran + port + prof)
# =====================================================
def merge_data(df_tran, df_port, df_prof):
    df_tran = df_tran.copy()
    df_port = df_port.copy()
    df_prof = df_prof.copy()

    df = df_tran.merge(df_port, on='offer_id', how='left')
    df = df.merge(df_prof, on='customer_id', how='left')

    print("Before merge:", df_tran.shape)
    print("After merge:", df.shape)
    return df


# =====================================================
# 6. Save Data
# =====================================================
def save_data(df, filename="preprocessed_fin.csv"):
    df.to_csv(filename, index=False)
    print(f"✅ {filename} 저장 완료")


# =====================================================
# 7. Run Pipeline
# =====================================================
def run_pipeline():
    pd, np, plt, ast = load_libraries()
    df_port, df_prof, df_tran, df_menu = load_data(pd)

    df_port = preprocess_portfolio(df_port, np)
    df_tran = preprocess_transcript(df_tran, ast)
    df_prof = preprocess_profile(df_prof, pd, np)

    df = merge_data(df_tran, df_port, df_prof)
    save_data(df)

    return df


# =====================================================
# 실행
# =====================================================
df = run_pipeline()
df.head()

프로모션 제공 데이터 크기: (10, 6)
고객정보 데이터 크기: (17000, 5)
제공 프로모션 데이터 크기: (306534, 4)
메뉴 정보 데이터 크기: (195, 12)
이벤트 미인지 완료: 4855건 / 전체 완료: 33579건 (14.5%)
✅ df_tran 전처리 완료
Before merge: (306534, 8)
After merge: (306534, 24)
✅ preprocessed_fin.csv 저장 완료


,person,event,time,amount,offer_id,day,customer_id,viewed_before_complete,reward,difficulty,...,ch_mobile,ch_social,channel_count,reward_ratio,offer_strength,gender,age,became_member_on,income,is_profile_missing
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,NaN,9b98b8c7a33c4b65b9aebfe6a799e6d9,0,78afa995795e4d85b5d9ceeca43f5fef,NaN,5.00,5.00,...,1.00,0.00,3.00,1.00,0.00,F,75.00,2017-05-09,100000.00,0
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,NaN,0b1e1539f2cc45b7b9fa7c272da2e1d7,0,a03223e636434f42ac4c3df47e8bac43,NaN,5.00,20.00,...,0.00,0.00,2.00,0.25,-15.00,Unknown,NaN,2017-08-04,0.00,1
2,e2127556f4f64592b11af22de27a7932,offer received,0,NaN,2906b810c7d4411798c6938adc9daaa5,0,e2127556f4f64592b11af22de27a7932,NaN,2.00,10.00,...,1.00,0.00,3.00,0.20,-8.00,M,68.00,2018-04-26,70000.00,0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,NaN,fafdcd668e3743c1bb461111dcafc2a4,0,8ec6ce2a7e7949b1bf142def7d0e0586,NaN,2.00,10.00,...,1.00,1.00,4.00,0.20,-8.00,Unknown,NaN,2017-09-25,0.00,1
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,NaN,4d5c57ea9a6940dd891ad53e9dbe8da0,0,68617ca6246f4fbc85e91a2a49552598,NaN,10.00,10.00,...,1.00,1.00,4.00,1.00,0.00,Unknown,NaN,2017-10-02,0.00,1


In [4]:
df.columns

Index(['person', 'event', 'time', 'amount', 'offer_id', 'day', 'customer_id',
       'viewed_before_complete', 'reward', 'difficulty', 'duration',
       'offer_type', 'ch_web', 'ch_email', 'ch_mobile', 'ch_social',
       'channel_count', 'reward_ratio', 'offer_strength', 'gender', 'age',
       'became_member_on', 'income', 'is_profile_missing'],
      dtype='str')